# Data Loading

This notebook downloads, extracts, and provides an initial inspection of the FAR-Trans dataset.
It covers all seven data files: transactions, customer information, asset information, markets, close prices, limit prices, and questionnaires.

In [ ]:
from pathlib import Path
import zipfile
import urllib.request
import shutil
import pandas as pd


def download_and_prepare_far_trans(root: Path) -> None:
    """
    Download, extract, and prepare the FAR-Trans dataset.

    The function:
    - Downloads the dataset if not present
    - Extracts contents into the root directory
    - Flattens nested FAR-Trans folder if present
    - Overwrites existing files safely
    - Removes the zip file after extraction
    """
    root.mkdir(parents=True, exist_ok=True)

    zip_url = "https://researchdata.gla.ac.uk/1658/1/FAR-Trans.zip"
    zip_path = root / "FAR-Trans.zip"

    if not zip_path.exists():
        urllib.request.urlretrieve(zip_url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)

    nested_folder = root / "FAR-Trans"

    if nested_folder.exists() and nested_folder.is_dir():
        for item in nested_folder.iterdir():
            destination = root / item.name

            if destination.exists():
                if destination.is_file():
                    destination.unlink()
                else:
                    shutil.rmtree(destination)

            shutil.move(str(item), root)

        shutil.rmtree(nested_folder)

    zip_path.unlink(missing_ok=True)


def find_file(root: Path, filename: str) -> Path:
    """
    Locate a file inside the dataset root directory.

    Args:
        root: Root dataset directory.
        filename: Name of the file to locate.

    Returns:
        Path to the located file.

    Raises:
        FileNotFoundError: If file cannot be found.
    """
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(f"Couldn't find {filename} under {root}")

    if len(matches) > 1:
        print(f"Warning: multiple matches for {filename}, using {matches[0]}")

    return matches[0]


def load_far_trans_dataset(root: Path) -> dict[str, pd.DataFrame | str]:
    """
    Load all FAR-Trans dataset files into pandas objects.

    Args:
        root: Root dataset directory.

    Returns:
        Dictionary containing loaded DataFrames and questionnaire text.
    """
    files = [
        "transactions.csv",
        "customer_information.csv",
        "asset_information.csv",
        "markets.csv",
        "close_prices.csv",
        "limit_prices.csv",
        "questionnaires.csv",
    ]

    paths = {fn: find_file(root, fn) for fn in files}

    data = {
        "transactions": pd.read_csv(
            paths["transactions.csv"], parse_dates=["timestamp"]
        ),
        "customers": pd.read_csv(
            paths["customer_information.csv"],
            parse_dates=["lastQuestionnaireDate", "timestamp"],
        ),
        "assets": pd.read_csv(
            paths["asset_information.csv"], parse_dates=["timestamp"]
        ),
        "markets": pd.read_csv(paths["markets.csv"]),
        "close_prices": pd.read_csv(
            paths["close_prices.csv"], parse_dates=["timestamp"]
        ),
        "limit_prices": pd.read_csv(
            paths["limit_prices.csv"], parse_dates=["minDate", "maxDate"]
        ),
        "questionnaire": paths["questionnaires.csv"].read_text(
            encoding="utf-8", errors="replace"
        ),
    }

    return data

In [ ]:
ROOT = Path("../data")

download_and_prepare_far_trans(ROOT)

dataset = load_far_trans_dataset(ROOT)

transactions = dataset["transactions"]
customers = dataset["customers"]
assets = dataset["assets"]
markets = dataset["markets"]
limit_prices = dataset["limit_prices"]
questionnaire = dataset["questionnaire"]

In [ ]:
customers.head(3)

In [ ]:
transactions.head(3)

In [ ]:
assets.head(3)

In [ ]:
markets.head(3)

In [ ]:
limit_prices.head(3)